<a href="https://colab.research.google.com/github/Sagaustus/adh-group-projects/blob/main/group-05-slave-voyages/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eighteen Thousand Captains, No Names

### The person as absent category in the Trans-Atlantic Slave Trade Database

**Group 5 · working chapter draft**

---

**Before you run anything.**

This dataset records the forced transportation of millions of people. The schema you
are about to analyse was built from commercial records — manifests, insurance
documents, port registers — kept by the people who organised that transportation.

Three things follow, and they are methodological as much as ethical:

1. **The schema's language is evidence, not vocabulary.** It uses words like *cargo*
   and *shipment*. Quote them; do not adopt them. Write *enslaved people*, never
   *slaves* as a noun for persons.
2. **Every number is people.** `n_slaves_arrived` is not a quantity of goods, and a
   chapter that treats it as a routine variable will read as though it does.
3. **The silences are the subject.** You are not filling gaps in an archive. You are
   analysing what a record-keeping system was built to see, and what it was built not
   to.

The analysis below is careful arithmetic. The care is the point.

---

**The thesis you are testing.** This database names 18,895 individual captains. It
names no enslaved person at all. That asymmetry is not an accident of survival — it is
the faithful reproduction, in a modern research infrastructure, of a commercial
standard designed to track property and liability.

In [ ]:
# Setup — run this first. Nothing to upload.
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

URL = "https://raw.githubusercontent.com/Sagaustus/adh-dh-datasets/main/datasets/07_slave_voyages/data.csv"
df = pd.read_csv(URL)
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(df.dtypes.to_string())

## Step 1 · Frame

| | Question | Method |
|---|---|---|
| **Descriptive** | Which entities does this schema individuate, and which does it aggregate? | Field-level comparison |
| **Analytical** | Is the absence of arrival counts random, and what does its structure do to published totals? | Logistic regression on missingness, then sensitivity analysis |

The analytical question matters because **every published estimate of the volume of
the trade rests on imputation across exactly this gap.**

## Step 2 · Absence audit

Do this first. For this chapter the absence is the entire finding.

In [ ]:
print("WHO IS INDIVIDUATED, AND WHO IS COUNTED\n")
print(f"{'field':<22}{'present':>10}{'distinct values':>18}")
print("-" * 52)
for c in df.columns:
    print(f"{c:<22}{df[c].notna().mean():>9.1%}{df[c].nunique():>18,}")
print()
print(f"captains recorded by name : {df['captains_name'].notna().mean():>6.1%}  "
      f"({df['captains_name'].nunique():,} distinct individuals)")
print(f"enslaved people recorded by name : {0:>4.1%}  (no such field exists)")
print()
print("The people carried appear only as `n_slaves_arrived`, a single integer per")
print(f"voyage, present for {df['n_slaves_arrived'].notna().mean():.1%} of voyages.")
print()
print(f"Sum of that column across the dataset: {df['n_slaves_arrived'].sum():,.0f} people.")
print("Roughly half the voyages contribute nothing to it, because nobody wrote the")
print("number down.")

### What the schema has no field for

- The name of any enslaved person
- Age, gender or origin community of any enslaved person
- Deaths during the crossing, as distinct from the arrival count
- Resistance, revolt or escape
- Anything at all about the people after arrival

The full SlaveVoyages database does carry some demographic breakdowns — men, women,
boys, girls — which **this export drops**. That is worth a paragraph in your chapter:
a derivation decision, made by whoever prepared this extract, removed the only
variables that could support a demographic question.

**YOUR DECISION.** Which absence most limits your questions? Two sentences.

## Step 3 · Describe

The variable at the centre of the analytical question is the *presence* of the
arrival count — so describe the missingness itself, not just the values.

In [ ]:
df["count_recorded"] = df["n_slaves_arrived"].notna()

print("arrival count recorded, by century\n")
print(f"{'century':>9}{'voyages':>10}{'recorded':>11}{'missing':>10}")
print("-" * 42)
for cent, g in df.groupby("century"):
    print(f"{cent:>9}{len(g):>10,}{g['count_recorded'].mean():>10.1%}"
          f"{1-g['count_recorded'].mean():>10.1%}")
print()
print("That is a gradient, not noise. Record-keeping improved across the period,")
print("so the archive is systematically thinner for the earlier trade.")
print()
vals = df["n_slaves_arrived"].dropna()
print(f"where recorded: median {vals.median():,.0f}, mean {vals.mean():,.1f}, "
      f"range {vals.min():,.0f}-{vals.max():,.0f}")
print()
print("Median and mean differ, so the distribution is skewed. Which you choose")
print("changes every total you compute — Step 5 makes that consequence explicit.")

## Step 4 · Compare

The two groups that matter are the voyages the archive counted and the voyages it did
not. If they differ systematically, the missing data is not missing at random and no
simple average can stand in for it.

In [ ]:
rec = df[df["count_recorded"]]
unrec = df[~df["count_recorded"]]

print(f"{'':<30}{'counted':>12}{'not counted':>14}")
print("-" * 58)
print(f"{'voyages':<30}{len(rec):>12,}{len(unrec):>14,}")
print(f"{'median year of arrival':<30}{rec['year_arrival'].median():>12.0f}"
      f"{unrec['year_arrival'].median():>14.0f}")
print(f"{'captain named':<30}{rec['captains_name'].notna().mean():>11.1%}"
      f"{unrec['captains_name'].notna().mean():>13.1%}")
print(f"{'ship named':<30}{rec['ship_name'].notna().mean():>11.1%}"
      f"{unrec['ship_name'].notna().mean():>13.1%}")
print(f"{'port of arrival recorded':<30}{rec['port_arrival'].notna().mean():>11.1%}"
      f"{unrec['port_arrival'].notna().mean():>13.1%}")
print()
print("top ports of arrival, counted voyages:")
for pt, n in rec["port_arrival"].value_counts().head(4).items():
    print(f"   {n:>6,}  {str(pt)[:50]}")
print()
print("top ports of arrival, uncounted voyages:")
for pt, n in unrec["port_arrival"].value_counts().head(4).items():
    print(f"   {n:>6,}  {str(pt)[:50]}")

## Step 5 · Test

First: is the missingness random? Then the consequence — what it does to the total.

In [ ]:
import statsmodels.api as sm

model_df = df.dropna(subset=["port_arrival"]).copy()
TOP_PORTS = 10
top_ports = model_df["port_arrival"].value_counts().head(TOP_PORTS).index
model_df["port"] = np.where(model_df["port_arrival"].isin(top_ports),
                            model_df["port_arrival"], "other")
model_df["year_c"] = model_df["year_arrival"] - model_df["year_arrival"].mean()

X = pd.get_dummies(model_df[["port"]], drop_first=True).astype(float)
X["year_c"] = model_df["year_c"].values
X = sm.add_constant(X)
y = model_df["count_recorded"].astype(int).values

fit = sm.Logit(y, X).fit(disp=0)
print(f"n = {len(y):,}   pseudo R-squared = {fit.prsquared:.3f}   "
      f"converged = {fit.mle_retvals['converged']}\n")

res = pd.DataFrame({"coef": fit.params, "p": fit.pvalues})
res["odds_ratio"] = np.exp(res["coef"])
res = res[res.index != "const"].sort_values("coef", ascending=False)
print(res.to_string(float_format=lambda v: f"{v:.3f}"))
print()
sig = int((res["p"] < 0.05).sum())
print(f"{sig} of {len(res)} predictors reach p < 0.05.")
print()
print("Whether a voyage's arrival count survives depends on when it sailed and where")
print("it landed. The data is NOT missing at random, so an overall mean is not a")
print("valid stand-in for the gap.")

### The sensitivity analysis

Here is the step that separates a careful chapter from a careless one.

The column sums to a specific number. That number is **only the voyages that were
counted**. Any statement about the total volume of the trade requires assuming
something about the other half — so state the assumption, vary it, and report a range
rather than a figure.

In [ ]:
observed_total = df["n_slaves_arrived"].sum()
n_missing = int((~df["count_recorded"]).sum())
vals = df["n_slaves_arrived"].dropna()

scenarios = {
    "counted voyages only (no imputation)": 0,
    "missing = overall median": vals.median() * n_missing,
    "missing = overall mean": vals.mean() * n_missing,
    "missing = century-specific median": sum(
        (~g["count_recorded"]).sum() * g["n_slaves_arrived"].median()
        for _, g in df.groupby("century") if g["n_slaves_arrived"].notna().any()),
    "missing = 25th percentile (conservative)": vals.quantile(.25) * n_missing,
    "missing = 75th percentile (high)": vals.quantile(.75) * n_missing,
}

print(f"voyages with no recorded arrival count: {n_missing:,} "
      f"({n_missing/len(df):.1%})\n")
print(f"{'assumption about the uncounted voyages':<44}{'implied total':>16}")
print("-" * 62)
for name, added in scenarios.items():
    print(f"{name:<44}{observed_total + added:>16,.0f}")
print()
lo = observed_total + min(scenarios.values())
hi = observed_total + max(scenarios.values())
print(f"range across these assumptions: {lo:,.0f} to {hi:,.0f}")
print(f"the highest is {hi/lo:.2f}x the lowest")
print()
print("Report the range and the assumptions, never a single total. A number given")
print("without its imputation assumption claims a precision the archive does not")
print("have — and here the choice of assumption moves the answer by millions.")

**YOUR DECISION.** Which assumption would you defend, and why?

There is no correct answer. The century-specific median is more defensible than the
overall median because the missingness is patterned by century — but it still assumes
uncounted voyages resembled counted ones from the same period, and Step 4 gave you
reason to doubt that.

Whatever you choose, the chapter must say what you chose and show the range.

## Step 6 · Show

One chart. The missingness gradient is the analytical finding, so plot it against the
volume of voyages — a reader needs to see that the worst-recorded centuries are also
the least numerous, which is why the gap is easy to overlook.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

by_cent = df.groupby("century").agg(
    voyages=("voyage_id", "size"),
    recorded=("count_recorded", "mean")).reset_index()

x = np.arange(len(by_cent))
ax1.bar(x, by_cent["voyages"], color="#d8d3ca", label="voyages recorded")
ax1.set_xticks(x); ax1.set_xticklabels([f"{c}th" for c in by_cent["century"]])
ax1.set_ylabel("voyages in the dataset")
ax1.set_xlabel("century")
ax1b = ax1.twinx()
ax1b.plot(x, by_cent["recorded"] * 100, "-o", color="#9c2c1f", lw=2,
          label="% with an arrival count")
ax1b.set_ylabel("voyages with an arrival count (%)", color="#9c2c1f")
ax1b.set_ylim(0, 100)
ax1.set_title("The archive thins as it goes back")

order = list(scenarios)
totals = [observed_total + scenarios[k] for k in order]
ax2.barh([k[:34] for k in order][::-1], totals[::-1], color="#2b5c50")
ax2.axvline(observed_total, ls="--", color="#9c2c1f", lw=1.5,
            label=f"counted only ({observed_total:,.0f})")
ax2.set_xlabel("implied total people disembarked")
ax2.set_title("The total depends on an assumption")
ax2.legend(loc="lower right", fontsize=8)
ax2.tick_params(labelsize=8)

plt.tight_layout()
print(f"CAPTION. All {len(df):,} voyages in the dataset; no filtering applied.")
print("Left: number of voyages recorded per century (bars, left axis) against the")
print("proportion carrying an arrival count (line, right axis). Right: implied total")
print("people disembarked under six stated assumptions about the "
      f"{n_missing:,} voyages")
print("with no recorded count. The dashed line is the sum of recorded counts alone.")

## Step 7 · The qualitative half

Two seams, and they belong together. The database individuates one party and
aggregates the other, so read what it says about the party it named.

In [ ]:
# Captain careers — reconstructible for the named party, and only for them.
careers = df["captains_name"].value_counts()
print(f"{df['captains_name'].nunique():,} distinct captains named")
print(f"{int((careers > 1).sum()):,} made more than one recorded voyage")
print(f"most voyages by one captain: {careers.max()}\n")

print("The ten most frequently recorded captains:")
for name, n in careers.head(10).items():
    voyages = df[df["captains_name"] == name]
    yrs = f"{int(voyages['year_arrival'].min())}-{int(voyages['year_arrival'].max())}"
    carried = voyages["n_slaves_arrived"].sum()
    known = int(voyages["n_slaves_arrived"].notna().sum())
    print(f"   {str(name)[:34]:<36}{n:>3} voyages  {yrs}   "
          f"{carried:>7,.0f} people recorded ({known}/{n} voyages)")
print()
print("A named man's career across decades can be reconstructed from this database.")
print("Not one person he transported can be. Put those two sentences next to each")
print("other in your chapter — they are the argument.")

In [ ]:
# Ship names — how the trade named itself.
from collections import Counter
import re

names = df["ship_name"].dropna()
print(f"{len(names):,} voyages carry a ship name; {names.nunique():,} distinct names\n")
print("The twenty commonest:")
for nm, n in names.value_counts().head(20).items():
    print(f"   {n:>4}  {nm}")
print()

words = Counter()
for nm in names:
    words.update(re.findall(r"[A-Za-zÀ-ÿ]{3,}", str(nm)))
print("Commonest words across all ship names:")
for w, n in words.most_common(18):
    print(f"   {n:>5}  {w}")

### What to do with the ship names

Read that list. Vessels carrying people as property were named for **women**
(*Mary*, *Nancy*, *Betsey*), for **saints and devotions** (*Nossa Senhora do
Rosário*, *Santo Antônio*), for **abstractions** (*Hope*, *Friendship*, *Liberty*),
and for **the continent itself** (*Africa*).

That is a self-representation, and it is available from a single column. Set it
against the schema's refusal to name anyone aboard, and you have the chapter's most
quotable page.

### The coding scheme

Take the fifty commonest ship names and code each one.

| Label | Definition |
|---|---|
| `PERSONAL_FEMALE` | A woman's given name |
| `PERSONAL_MALE` | A man's given name |
| `DEVOTIONAL` | A saint, a religious title, a devotion |
| `VIRTUE_ABSTRACT` | Hope, Friendship, Industry, Liberty |
| `PLACE` | A port, region, country or continent |
| `OTHER` | Anything else — record what |

**Two coders, independently.** Then report κ. The `VIRTUE_ABSTRACT` and `DEVOTIONAL`
categories are where the argument lives, and they are also where careful readers
disagree — which is exactly why the agreement figure is worth having.

In [ ]:
from sklearn.metrics import cohen_kappa_score

top50 = names.value_counts().head(50)
print("The fifty to code:\n")
for i, (nm, n) in enumerate(top50.items(), 1):
    end = "\n" if i % 2 == 0 else "   "
    print(f"{i:>3}. {str(nm)[:30]:<32}({n:>3})", end=end)
print()

# Replace with your real codes. Ten shown here so the mechanics run.
coder_a = ["PERSONAL_FEMALE","PERSONAL_FEMALE","DEVOTIONAL","DEVOTIONAL","PLACE",
           "DEVOTIONAL","PERSONAL_FEMALE","VIRTUE_ABSTRACT","PERSONAL_MALE","DEVOTIONAL"]
coder_b = ["PERSONAL_FEMALE","PERSONAL_FEMALE","DEVOTIONAL","DEVOTIONAL","PLACE",
           "DEVOTIONAL","PERSONAL_FEMALE","DEVOTIONAL","PERSONAL_MALE","VIRTUE_ABSTRACT"]

kappa = cohen_kappa_score(coder_a, coder_b)
raw = float(np.mean([x == y for x, y in zip(coder_a, coder_b)]))
print(f"\nraw agreement {raw:.2f}   Cohen's kappa {kappa:.3f}  (n = {len(coder_a)})")
print()
print("Disagreements:")
for i, (x, y) in enumerate(zip(coder_a, coder_b), 1):
    if x != y:
        print(f"   name {i}: {x} vs {y}")
print()
print("Code all fifty for the chapter. The distribution across categories is the")
print("finding; the kappa is what makes it reportable rather than impressionistic.")

## Step 8 · Limits

**What this analysis supports**

- Statements about this extract of the SlaveVoyages database, on its stated date.
- The asymmetry between named captains and unnamed enslaved people, as a property of
  the schema.
- That the absence of arrival counts is patterned by period and port, at the effect
  sizes reported.
- A range of implied totals under stated assumptions.

**What it does not support**

- Any single figure for the volume of the trade. You have a range and a set of
  assumptions, and the chapter must present them as such.
- Any claim about mortality. The dataset records arrivals, not departures or deaths;
  the difference is a different variable that this export does not carry.
- Any demographic claim. Age and gender breakdowns exist upstream and were dropped in
  this derivation.
- Any claim about individual enslaved people. There is no such record here, and that
  absence is the chapter's subject rather than a gap to be filled by inference.
- Any claim about voyages absent from the database entirely. Unrecorded voyages are
  invisible to this analysis by definition.

**YOUR DECISION.** Add two more sentences this data does not support.

---

# Step 9 · The chapter template

Fill the gaps, then rewrite in your own voice. **Length: 6,000–8,000 words.**

A note on register for this chapter specifically: write plainly. The material does not
need heightening, and rhetorical escalation reads as distrust of the evidence.

---

## §1 Introduction — *about 800 words*

> The Trans-Atlantic Slave Trade Database is among the most used datasets in
> quantitative history, underpinning ______________ . This chapter examines
> **_____ voyages** recorded between **_____ and _____**, and argues that
> ______________ .

*Write last.*

---

## §2 The standard and its critics — *about 1,200 words*

The database's construction, its variable schema, and the historiography of counting
the trade. Then the critical literature on archival silence — Trouillot on the
production of history, Hartman on the limits of the archive, Johnson on the ethics of
narrating what the record refuses.

> The database records voyages through variables including ______________ .
> Its unit of analysis is ______________ , which means ______________ .
> Scholars have argued ______________ (cite).

---

## §3 Data — *about 900 words*

> The extract comprises **_____ voyages** arriving between **_____ and _____**.
> Captains are named for **_____%** of voyages, comprising **_____** distinct
> individuals. Arrival counts are present for **_____%**. No field records the
> name, age, gender or origin community of any enslaved person.

Then the derivation note, which matters:

> Demographic breakdowns present in the full database are absent from this extract,
> a decision taken at ______________ .

---

## §4 Method — *about 700 words*

> The presence of an arrival count was modelled by logistic regression on
> ______________ , to test whether absence is random.
> Implied totals were computed under **six stated assumptions** about uncounted
> voyages, reported as a range rather than a point estimate, because ______________ .
> The fifty commonest ship names were coded independently by two readers.

---

## §5 Findings — *about 1,800 words*

**§5.1 The schema individuates one party and aggregates the other**

> Captains are named on **_____%** of voyages (**_____** individuals). Enslaved
> people appear only as a count, present for **_____%** of voyages.

**§5.2 The absence is patterned, not random**

> Arrival counts survive for **_____%** of sixteenth-century voyages and **_____%**
> of nineteenth-century ones. Regression on period and port of arrival gives
> ______________ .

**§5.3 What that does to the total**

> Recorded counts sum to **_____**. Under six stated assumptions the implied total
> ranges from **_____ to _____**, a factor of **_____**.

**§5.4 How the trade named itself** *(qualitative)*

> Of the fifty commonest vessel names, **_____** were personal names, **_____**
> devotional and **_____** abstractions (κ = _____). The commonest were
> ______________ .

**Figure 1** after §5.2, captioned from Step 6.

---

## §6 Discussion — *about 1,400 words*

> The asymmetry is best understood as ______________ rather than as ______________ .
> For quantitative history, the consequence is ______________ .
> For descendant communities, ______________ .

The counter-practice, and here it is unusually concrete:

> The *African Origins* project responds by ______________ , drawing on
> ______________ . Its limits are ______________ .

That project collects African names from registers of liberated Africans, inverting
precisely the silence you have measured. Comparing the two data models directly is the
strongest move available to this chapter.

---

## §7 Limitations · §8 Conclusion · §9 References · §10 Data and code

In [ ]:
print("=" * 74)
print("DRAFT SENTENCES — your numbers already placed")
print("=" * 74)

cent = df.groupby("century")["count_recorded"].mean()

print("""
§3 DATA
  The extract comprises {n:,} voyages arriving between {y0} and {y1}. A captain is
  named for {cp:.1%} of voyages, comprising {cn:,} distinct individuals. An arrival
  count is present for {ap:.1%}. No field records the name, age, gender or origin
  community of any enslaved person; they appear only as an aggregate count per
  voyage. Demographic breakdowns carried by the full database are absent from this
  extract.
""".format(n=len(df), y0=int(df["year_arrival"].min()), y1=int(df["year_arrival"].max()),
           cp=df["captains_name"].notna().mean(), cn=df["captains_name"].nunique(),
           ap=df["n_slaves_arrived"].notna().mean()))

print("""§5.1 THE SCHEMA INDIVIDUATES ONE PARTY AND AGGREGATES THE OTHER
  The database names {cn:,} individual captains across {cp:.1%} of voyages, and
  {rep:,} of them made more than one recorded voyage; the most frequently recorded
  made {mx}. It names no enslaved person. The people transported are represented by
  a single integer per voyage, absent for {miss:.1%} of records.
""".format(cn=df["captains_name"].nunique(), cp=df["captains_name"].notna().mean(),
           rep=int((careers > 1).sum()), mx=int(careers.max()),
           miss=1 - df["n_slaves_arrived"].notna().mean()))

print("""§5.2 THE ABSENCE IS PATTERNED, NOT RANDOM
  An arrival count survives for {c16:.0%} of sixteenth-century voyages and {c19:.0%}
  of nineteenth-century ones. Logistic regression on period and port of arrival gives
  pseudo R-squared {r:.3f} (n = {mn:,}), with {sig} of {tot} predictors reaching
  p < 0.05. The gap is therefore structured by when a voyage sailed and where it
  landed, and cannot be treated as missing at random.
""".format(c16=cent.get(16, float("nan")), c19=cent.get(19, float("nan")),
           r=fit.prsquared, mn=len(y), sig=sig, tot=len(res)))

print("""§5.3 WHAT THAT DOES TO THE TOTAL
  Recorded arrival counts sum to {obs:,.0f} people. Under six stated assumptions
  about the {miss:,} voyages with no recorded count, the implied total ranges from
  {lo:,.0f} to {hi:,.0f} - a factor of {f:.2f}. No single figure is reportable
  without its assumption.
""".format(obs=observed_total, miss=n_missing, lo=lo, hi=hi, f=hi/lo))
print("=" * 74)
print("Left for you: what it MEANS, what it cannot support, the counter-practice.")

### Three mistakes that sink first chapters

**Reporting a single total.** The column sums to a number. That number is the counted
voyages only, and presenting it as the volume of the trade is the error this whole
chapter exists to expose. Report the range.

**Adopting the schema's vocabulary.** *Cargo*, *shipment*, *slaves* as a noun for
people. Quote them as evidence of how the records were kept; do not write in them.

**Treating the silence as a data problem.** It is a finding. A chapter that spends its
length lamenting incompleteness has not analysed the incompleteness — and the
structure of the absence, which you have measured, is the analysis.

---

# Step 10 · Dividing the work

More than five people, one chapter. Divide by **expertise**, not by paragraph count.

## The roles

| # | Role | Owns | Expertise it draws on | Hands over |
|---|---|---|---|---|
| 1 | **Corpus &amp; schema** | §3, absence audit | Archival history, database documentation | The variable audit and the derivation note |
| 2 | **Analysis** | §4, §5.1–5.3 | Statistics, computation | The missingness model and the sensitivity range |
| 3 | **Coder A** | §5.4, jointly | Onomastics, maritime and religious history | 50 independently assigned codes |
| 4 | **Coder B** | §5.4, jointly | Onomastics, maritime and religious history | 50 independently assigned codes |
| 5 | **Theory &amp; historiography** | §2, §6 | Trouillot, Hartman, Johnson; the archive-silence literature | The argument the chapter joins |
| 6 | **Counter-practice** | §6, part of §2 | Public history, descendant-community projects | African Origins, and what it does differently |
| 7 | **Integration editor** | §1, §7, register | Editorial judgement | One voice, and control of tone |

Role 7 carries more than usual here. **Register is a methodological matter in this
chapter**, and one section written in a heightened voice will undermine the restraint
of the rest.

Role 6 is separate from role 5 because the counter-practice needs its own research.
*African Origins* is not a citation; it is a data model, and comparing it directly
against this schema is the chapter's strongest move.

## Why coders 3 and 4 are two people

A **method requirement.** Whether *Hope* is an abstraction or a vessel-naming
convention, whether *Nossa Senhora do Rosário* counts as devotional or as a personal
name — careful readers differ. Kappa converts that difference into a reportable figure
rather than an argument in a meeting.

## The order things happen in

```
   Corpus & schema ──┐
                     ├──► Analysis ──┐
   Coders A + B ─────┘               ├──► Integration
                                     │
   Theory & historiography ──────────┤
   Counter-practice ─────────────────┘
```

## Combining the drafts

**One person edits for voice, and everyone accepts the edit.**

**Agree the terms in writing before drafting.** For this chapter that is not a style
preference. Decide, and write it down: *enslaved people* not *slaves*; *voyage* not
*shipment*; *people disembarked* not *cargo landed*. Where the schema's own words
appear, they go in quotation marks as evidence.

## Declaring who did what

Use **CRediT**.

In [ ]:
TEAM = {
    "Corpus & schema":          ("________________", "data curation, investigation"),
    "Analysis":                 ("________________", "formal analysis, software, methodology"),
    "Coder A":                  ("________________", "investigation, validation"),
    "Coder B":                  ("________________", "investigation, validation"),
    "Theory & historiography":  ("________________", "conceptualisation, writing - original draft"),
    "Counter-practice":         ("________________", "investigation, writing - original draft"),
    "Integration editor":       ("________________", "writing - review and editing, supervision"),
}
print("AUTHOR CONTRIBUTIONS (CRediT)")
print()
for role, (name, credit) in TEAM.items():
    print(f"  {name}: {credit}.")
    print(f"      [{role}]")
print()
print("Fill in names, delete the bracketed labels, place after the conclusion.")
print("Agree authorship order EARLY.")

### What goes wrong, and how to see it coming

**The register slips.** The commonest failure on this material is a section that
reaches for effect. Role 7 owns tone, and has authority to rewrite.

**A single total gets into the abstract.** It will be the most quoted sentence in the
chapter and the easiest to attack. Range, always.

**The counter-practice becomes a footnote.** Role 6 exists so that African Origins gets
analysed as a data model rather than mentioned in passing.

**The coders talk.** Fatal to the kappa.

**Nobody owns the ending.** Role 7 owns §8 and the decision that it is finished.

## What to hand in

1. **One page**: the finding, the method, the uncertainty, and the limits list.
2. **One chart**, captioned, saying what you filtered.
3. **The coding sheet** for all fifty ship names, with both coders' labels and κ.
4. **The sensitivity table**, with every assumption stated.

### Turning this into the chapter

Your spine is the strongest of the five, and the plainest: a database that names
18,895 captains and no enslaved person; an absence that is structured by century and
port rather than random; and a total that moves by millions depending on an assumption
most publications never state.

What it needs from you is **§6** — what follows for quantitative history when its
foundational dataset inherits a commercial standard's decisions about who counts as an
individual — and the counter-practice. *African Origins* built a data model to hold
names the manifests refused to record. Comparing the two schemas directly is where
this chapter becomes a contribution rather than a critique.